In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

1. Data Loading

In [3]:

# List of file paths
files = [
    "BIO_1.csv","BIO_2.csv","BIO_3.csv","BIO_4.csv"
]

# Read and store each file
dfs = [pd.read_csv(f) for f in files]

In [4]:
# Merge into one DataFrame
merged_df = pd.concat(dfs, ignore_index=True)

In [5]:
merged_df.shape

(1861108, 6)

In [6]:
#to check if merging is done correctly
total_rows = sum(df.shape[0] for df in dfs)
print("Total rows before merging:", total_rows)
print("Rows after merging:", merged_df.shape[0])

Total rows before merging: 1861108
Rows after merging: 1861108


2. Data Understanding

Basic Overview of Data

In [7]:
merged_df.columns

Index(['date', 'state', 'district', 'pincode', 'bio_age_5_17', 'bio_age_17_'], dtype='str')

In [8]:
merged_df.head()

,date,state,district,pincode,bio_age_5_17,bio_age_17_
0,01-03-2025,Haryana,Mahendragarh,123029,280,577
1,01-03-2025,Bihar,Madhepura,852121,144,369
2,01-03-2025,Jammu and Kashmir,Punch,185101,643,1091
3,01-03-2025,Bihar,Bhojpur,802158,256,980
4,01-03-2025,Tamil Nadu,Madurai,625514,271,815


In [9]:
merged_df.tail()

,date,state,district,pincode,bio_age_5_17,bio_age_17_
1861103,29-12-2025,West Bengal,Uttar Dinajpur,733201,4,9
1861104,29-12-2025,West Bengal,Uttar Dinajpur,733213,0,1
1861105,29-12-2025,West Bengal,West Midnapore,721304,0,3
1861106,29-12-2025,West Bengal,West Midnapore,721451,2,0
1861107,29-12-2025,West Bengal,West Midnapore,721457,0,1


3. Data Cleaning

In [10]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1861108 entries, 0 to 1861107
Data columns (total 6 columns):
 #   Column        Dtype
---  ------        -----
 0   date          str  
 1   state         str  
 2   district      str  
 3   pincode       int64
 4   bio_age_5_17  int64
 5   bio_age_17_   int64
dtypes: int64(3), str(3)
memory usage: 85.2 MB


In [11]:
merged_df['date'] = pd.to_datetime(merged_df['date'], format='%d-%m-%Y')

In [12]:
merged_df.dtypes

date            datetime64[us]
state                      str
district                   str
pincode                  int64
bio_age_5_17             int64
bio_age_17_              int64
dtype: object

Check Missing Values

In [13]:
merged_df.isnull().sum()

date            0
state           0
district        0
pincode         0
bio_age_5_17    0
bio_age_17_     0
dtype: int64

Check Duplicate Rows

In [14]:
merged_df.duplicated().sum()

np.int64(94896)

In [15]:
merged_df[merged_df.duplicated()].head(10)


,date,state,district,pincode,bio_age_5_17,bio_age_17_
110000,2025-09-01,Chhattisgarh,Kondagaon,494229,0,1
110001,2025-09-01,Chhattisgarh,Kondagaon,494230,1,0
110002,2025-09-01,Chhattisgarh,Korba,495119,5,35
110003,2025-09-01,Chhattisgarh,Korba,495446,0,16
110004,2025-09-01,Chhattisgarh,Korba,495674,10,34
110005,2025-09-01,Chhattisgarh,Korba,495683,0,3
120068,2025-09-01,Assam,Sonitpur,784174,0,1
120069,2025-09-01,Assam,Sonitpur,784182,4,4
120070,2025-09-01,Assam,South Salmara Mankachar,783127,3,3
120071,2025-09-01,Assam,South Salmara Mankachar,783128,0,1


In [16]:
 merged_df = merged_df.drop_duplicates()

In [17]:
merged_df.duplicated().sum()

np.int64(0)

In [18]:
merged_df.shape

(1766212, 6)

The dataset initially contained 94,896 duplicate rows (~5% of the data). These were removed to prevent distortion in aggregated statistics and ensure accurate analysis.

Clean Text Columns

In [ ]:
#Remove spaces and standardize text
merged_df['state'] = merged_df['state'].str.strip().str.title()
merged_df['district'] = merged_df['district'].str.strip().str.title()

In [20]:
#Fix inconsistent names
merged_df['state'] = merged_df['state'].replace({
    'Westbengal':'West Bengal',
    'Chhatisgarh':'Chhattisgarh',
    'Jammu & Kashmir':'Jammu And Kashmir',
    'Daman & Diu':'Daman And Diu'
})

In [21]:
#to check inconsistent names remain
merged_df['state'].unique()

<StringArray>
[                                 'Haryana',
                                    'Bihar',
                        'Jammu And Kashmir',
                               'Tamil Nadu',
                              'Maharashtra',
                                  'Gujarat',
                                   'Odisha',
                              'West Bengal',
                                   'Kerala',
                                'Rajasthan',
                                   'Punjab',
                         'Himachal Pradesh',
                            'Uttar Pradesh',
                                    'Assam',
                              'Uttarakhand',
                           'Madhya Pradesh',
                                'Karnataka',
                           'Andhra Pradesh',
                                'Telangana',
                                      'Goa',
                                 'Nagaland',
                                'Jharkhan

In [22]:
merged_df['state'] = merged_df['state'].str.replace(r'\s+', ' ', regex=True)

In [23]:
merged_df['state'] = merged_df['state'].replace({
    'West Bangal': 'West Bengal',
    'Tamilnadu': 'Tamil Nadu',
    'Uttaranchal': 'Uttarakhand'
})

In [24]:
sorted(merged_df['state'].unique())

['Andaman & Nicobar Islands',
 'Andaman And Nicobar Islands',
 'Andhra Pradesh',
 'Arunachal Pradesh',
 'Assam',
 'Bihar',
 'Chandigarh',
 'Chhattisgarh',
 'Dadra & Nagar Haveli',
 'Dadra And Nagar Haveli',
 'Dadra And Nagar Haveli And Daman And Diu',
 'Daman And Diu',
 'Delhi',
 'Goa',
 'Gujarat',
 'Haryana',
 'Himachal Pradesh',
 'Jammu And Kashmir',
 'Jharkhand',
 'Karnataka',
 'Kerala',
 'Ladakh',
 'Lakshadweep',
 'Madhya Pradesh',
 'Maharashtra',
 'Manipur',
 'Meghalaya',
 'Mizoram',
 'Nagaland',
 'Odisha',
 'Orissa',
 'Pondicherry',
 'Puducherry',
 'Punjab',
 'Rajasthan',
 'Sikkim',
 'Tamil Nadu',
 'Telangana',
 'Tripura',
 'Uttar Pradesh',
 'Uttarakhand',
 'West Bengal']

In [25]:
merged_df['state'] = merged_df['state'].replace({
    'Andaman & Nicobar Islands': 'Andaman And Nicobar Islands',
    'Dadra & Nagar Haveli': 'Dadra And Nagar Haveli'
})

In [26]:
sorted(merged_df['state'].unique())

['Andaman And Nicobar Islands',
 'Andhra Pradesh',
 'Arunachal Pradesh',
 'Assam',
 'Bihar',
 'Chandigarh',
 'Chhattisgarh',
 'Dadra And Nagar Haveli',
 'Dadra And Nagar Haveli And Daman And Diu',
 'Daman And Diu',
 'Delhi',
 'Goa',
 'Gujarat',
 'Haryana',
 'Himachal Pradesh',
 'Jammu And Kashmir',
 'Jharkhand',
 'Karnataka',
 'Kerala',
 'Ladakh',
 'Lakshadweep',
 'Madhya Pradesh',
 'Maharashtra',
 'Manipur',
 'Meghalaya',
 'Mizoram',
 'Nagaland',
 'Odisha',
 'Orissa',
 'Pondicherry',
 'Puducherry',
 'Punjab',
 'Rajasthan',
 'Sikkim',
 'Tamil Nadu',
 'Telangana',
 'Tripura',
 'Uttar Pradesh',
 'Uttarakhand',
 'West Bengal']

In [27]:
merged_df['state'].nunique()

40

In [28]:
#as the output is not as expected so we furthur investigate
sorted(merged_df['state'].unique())

['Andaman And Nicobar Islands',
 'Andhra Pradesh',
 'Arunachal Pradesh',
 'Assam',
 'Bihar',
 'Chandigarh',
 'Chhattisgarh',
 'Dadra And Nagar Haveli',
 'Dadra And Nagar Haveli And Daman And Diu',
 'Daman And Diu',
 'Delhi',
 'Goa',
 'Gujarat',
 'Haryana',
 'Himachal Pradesh',
 'Jammu And Kashmir',
 'Jharkhand',
 'Karnataka',
 'Kerala',
 'Ladakh',
 'Lakshadweep',
 'Madhya Pradesh',
 'Maharashtra',
 'Manipur',
 'Meghalaya',
 'Mizoram',
 'Nagaland',
 'Odisha',
 'Orissa',
 'Pondicherry',
 'Puducherry',
 'Punjab',
 'Rajasthan',
 'Sikkim',
 'Tamil Nadu',
 'Telangana',
 'Tripura',
 'Uttar Pradesh',
 'Uttarakhand',
 'West Bengal']

In [29]:
merged_df['state'] = merged_df['state'].replace({
    'Dadra And Nagar Haveli': 'Dadra And Nagar Haveli And Daman And Diu',
    'Daman And Diu': 'Dadra And Nagar Haveli And Daman And Diu'
})

In [30]:
merged_df['state'].nunique()

38

In [32]:
for state in sorted(merged_df['state'].unique()):
    print(state)

Andaman And Nicobar Islands
Andhra Pradesh
Arunachal Pradesh
Assam
Bihar
Chandigarh
Chhattisgarh
Dadra And Nagar Haveli And Daman And Diu
Delhi
Goa
Gujarat
Haryana
Himachal Pradesh
Jammu And Kashmir
Jharkhand
Karnataka
Kerala
Ladakh
Lakshadweep
Madhya Pradesh
Maharashtra
Manipur
Meghalaya
Mizoram
Nagaland
Odisha
Orissa
Pondicherry
Puducherry
Punjab
Rajasthan
Sikkim
Tamil Nadu
Telangana
Tripura
Uttar Pradesh
Uttarakhand
West Bengal


In [33]:
for s in sorted(merged_df['state'].unique()):
    print(s)

Andaman And Nicobar Islands
Andhra Pradesh
Arunachal Pradesh
Assam
Bihar
Chandigarh
Chhattisgarh
Dadra And Nagar Haveli And Daman And Diu
Delhi
Goa
Gujarat
Haryana
Himachal Pradesh
Jammu And Kashmir
Jharkhand
Karnataka
Kerala
Ladakh
Lakshadweep
Madhya Pradesh
Maharashtra
Manipur
Meghalaya
Mizoram
Nagaland
Odisha
Orissa
Pondicherry
Puducherry
Punjab
Rajasthan
Sikkim
Tamil Nadu
Telangana
Tripura
Uttar Pradesh
Uttarakhand
West Bengal


In [34]:
official_states = [
"Andaman And Nicobar Islands","Andhra Pradesh","Arunachal Pradesh","Assam",
"Bihar","Chandigarh","Chhattisgarh","Dadra And Nagar Haveli And Daman And Diu",
"Delhi","Goa","Gujarat","Haryana","Himachal Pradesh","Jammu And Kashmir",
"Jharkhand","Karnataka","Kerala","Ladakh","Lakshadweep","Madhya Pradesh",
"Maharashtra","Manipur","Meghalaya","Mizoram","Nagaland","Odisha","Punjab",
"Rajasthan","Sikkim","Tamil Nadu","Telangana","Tripura","Uttar Pradesh",
"Uttarakhand","West Bengal","Puducherry"
]

set(merged_df['state']) - set(official_states)

{'Orissa', 'Pondicherry'}

In [35]:
merged_df['state'] = merged_df['state'].replace({
    'Orissa': 'Odisha',
    'Pondicherry': 'Puducherry'
})

In [36]:
merged_df['state'].nunique()

36

4. Data Analysis
